In [6]:
# -------- Config --------
DATADIR = "/work/gr-fe/bryan/data/SHCS/"
COHORT = ""
META_PICKLE = f"{DATADIR}/{COHORT}/02_processed/phenotype.processed.pkl"
OMICS = ["Proteomics" , "Metabolomics" , "PGS"]  # datExpr_{omic}.csv for each
RAW_OMICS_DIR = f"{DATADIR}/{COHORT}/01_raw/"
OUT_DIR = f"{DATADIR}/{COHORT}/02_processed"
OVERWRITE = True  # overwrite existing output pickles

# -------- Imports --------
import os
import numpy as np
import pickle
from pathlib import Path
import pandas as pd

# -------- Helpers --------
def load_meta_ids(meta_path: str) -> pd.Series:
    """
    Load a pickle expected to contain either:
      - a pandas DataFrame with column 'ID', or
      - a dict with key 'ID' (array-like)
    Returns a clean string Series of IDs (duplicates removed, NaNs dropped).
    """
    # Try pandas first (faster for DataFrame pickles), fall back to pickle.load
    meta = None
    try:
        meta = pd.read_pickle(meta_path)
    except Exception:
        with open(meta_path, "rb") as f:
            meta = pickle.load(f)

    if isinstance(meta, pd.DataFrame) and "ID" in meta.columns:
        ids = meta["ID"]
    elif isinstance(meta, dict) and "ID" in meta:
        ids = pd.Series(meta["ID"], name="ID")
    else:
        raise ValueError("Meta pickle must be a DataFrame with 'ID' column or a dict with key 'ID'.")

    ids = ids.dropna().astype(str)
    ids = ids[~ids.duplicated()].reset_index(drop=True)
    return ids

def orient_to_ids(df: pd.DataFrame, id_set: set) -> pd.DataFrame | None:
    """
    Ensure sample IDs are in the index; if they're in columns, transpose.
    If neither index nor columns contain any meta IDs, return None.
    """
    # Normalize types to str for safe matching
    df.index = df.index.map(str)
    df.columns = df.columns.map(str)

    idx_hit = len(id_set.intersection(df.index))
    col_hit = len(id_set.intersection(df.columns))

    if idx_hit == 0 and col_hit > 0:
        df = df.T
        idx_hit = len(id_set.intersection(df.index))

    if idx_hit == 0:
        return None
    return df

# -------- Run --------
out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

ids = load_meta_ids(META_PICKLE)
id_set = set(ids)

print(f"Loaded {len(ids)} unique meta IDs from: {META_PICKLE}")
print(f"Output dir: {out_dir}\n")

results = {}
for omic in OMICS:
    src = Path(RAW_OMICS_DIR) / f"datExpr_{omic}.csv"
    if not src.exists():
        print(f"[{omic}] SKIP - not found: {src}")
        continue

    try:
        df = pd.read_csv(src, index_col=0, dtype=str)
    except Exception as e:
        print(f"[{omic}] ERROR reading {src}: {e}")
        continue

    df_oriented = orient_to_ids(df, id_set)
    if df_oriented is None:
        print(f"[{omic}] SKIP - no overlap between meta IDs and {src.name} (rows or columns).")
        continue

    # Preserve meta ID order in the subset
    keep_ids = [i for i in ids if i in df_oriented.index]
    sub = df_oriented.loc[keep_ids].astype(np.float32) 

    out_path = out_dir / f"{omic}.pkl"
    if out_path.exists() and not OVERWRITE:
        print(f"[{omic}] Exists, not overwritten: {out_path} (shape={sub.shape})")
    else:
        with open(out_path, "wb") as f:
            pickle.dump({"expr": sub}, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"[{omic}] Saved {sub.shape} to {out_path}")

    results[omic] = sub

# Optional: quick peek at one modality
for omic in OMICS:
    if omic in results:
        display(results[omic].head())
        break


Loaded 2095 unique meta IDs from: /work/gr-fe/bryan/data/SHCS///02_processed/phenotype.processed.pkl
Output dir: /work/gr-fe/bryan/data/SHCS/02_processed

[Proteomics] Saved (987, 766) to /work/gr-fe/bryan/data/SHCS/02_processed/Proteomics.pkl
[Metabolomics] Saved (1307, 1930) to /work/gr-fe/bryan/data/SHCS/02_processed/Metabolomics.pkl
[PGS] Saved (1815, 3096) to /work/gr-fe/bryan/data/SHCS/02_processed/PGS.pkl


,A0A075B6H7,A0A075B6I0,A0A075B6I7,A0A075B6I9,A0A075B6J1,A0A075B6J9,A0A075B6K0,A0A075B6K0;A0A075B6K4,A0A075B6K0;P01717;P01718,A0A075B6K2,...,Q92496;Q92496-2,Q92954-3;Q92954-6,Q9BUN1;Q9BUN1-2,Q9BWP8;Q9BWP8-10;Q9BWP8-2;Q9BWP8-3;Q9BWP8-4;Q9BWP8-5;Q9BWP8-6;Q9BWP8-7;Q9BWP8-8;Q9BWP8-9,Q9HBI1;Q9HBI1-2;Q9HBI1-3,Q9NQ79;Q9NQ79-2,Q9UEW3;Q9UEW3-2,Q9Y2I9;Q9Y2I9-2;Q9Y2I9-3;Q9Y2I9-4,Q9Y490;Q9Y490-2,Q9Y490;Q9Y490-2;Q9Y4G6
ID,,,,,,,,,,,,,,,,,,,,,
30119,17.706413,17.204767,14.974404,14.821267,7.217602,16.093157,15.475765,15.575383,13.162805,14.720095,...,16.353861,15.296158,8.993970,12.569895,12.272435,12.972963,11.697124,12.918978,15.352096,13.391688
31593,18.794203,15.775344,9.906116,13.800585,14.167992,16.069101,10.335061,15.137912,12.347219,9.312797,...,16.929024,15.777438,6.107157,12.403109,11.026884,13.618512,10.925058,13.070668,14.437764,11.040124
25890,18.082550,15.671324,12.473639,13.791698,13.100969,16.046368,14.584240,15.083038,13.269654,6.948315,...,17.004341,15.250875,9.953929,11.657607,9.596213,13.492507,11.477224,14.339425,11.016101,12.314171
47196,17.169865,15.130647,15.023294,15.481746,14.632660,15.746859,14.949437,14.680860,13.243638,11.818766,...,16.206591,15.928973,14.753299,12.971766,11.051301,13.612262,11.165910,11.722240,14.588238,12.151222
17150,18.407547,16.061588,14.546513,9.456702,12.183483,16.217535,13.849149,15.494027,12.439260,13.054408,...,15.477974,14.894138,10.015727,12.168914,11.858830,13.648769,11.520417,14.833682,15.290628,13.186101
